In [1]:
# tests/run_extract_and_write.py
# -*- coding: utf-8 -*-
"""
IFC 読み込み → BDNS 抽出 → 空間要素抽出 → JSON 出力 → Brick Equipment RDF 出力
前提:
  - プロジェクトの src を sys.path に追加しトップレベルパッケージとして import する
  - ifc_ingestor.ingestor.load_ifc_bundle が利用可能であること
  - bdns_extractor.extractor.extract_bdns_tags, extract_spatial_elements が利用可能であること
  - rdf_writer.writer.write_equipments_turtle が利用可能であること
"""


'\nIFC 読み込み → BDNS 抽出 → 空間要素抽出 → JSON 出力 → Brick Equipment RDF 出力\n前提:\n  - プロジェクトの src を sys.path に追加しトップレベルパッケージとして import する\n  - ifc_ingestor.ingestor.load_ifc_bundle が利用可能であること\n  - bdns_extractor.extractor.extract_bdns_tags, extract_spatial_elements が利用可能であること\n  - rdf_writer.writer.write_equipments_turtle が利用可能であること\n'

In [2]:
import sys
from pathlib import Path
import json

In [22]:
# ============ プロジェクト設定 ============
PROJECT_ROOT = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project")
SRC_PATH = PROJECT_ROOT / "src"
IFC_PATH1 = PROJECT_ROOT / "data" / "ifc" / "raw" / "51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01.ifc"
IFC_PATH2 = 
OUT_DIR = PROJECT_ROOT / "data" / "output"

In [14]:
# sys.path 設定
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))
print("sys.path に追加された src:", SRC_PATH)

sys.path に追加された src: C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\src


In [15]:
# ============ モジュール import ============
from ifc_ingestor.ingestor import load_ifc_bundle
from bdns_extractor.extractor import extract_bdns_tags, extract_spatial_elements
from rdf_writer.writer import write_equipments_turtle

In [16]:
# ============ IFC 読み込み ============
bundle = load_ifc_bundle(str(IFC_PATH))
print("\n=== IFC Loaded ===")
print("schema:", getattr(bundle, "schema", None))
print("model type:", type(getattr(bundle, "model", None)))


=== IFC Loaded ===
schema: IFC2X3
model type: <class 'ifcopenshell.file.file'>


In [17]:
# ============ BDNS タグ抽出 ============
bdns_assets = extract_bdns_tags(bundle)
print("\n=== BDNS Extraction Result ===")
print("BDNS-tagged assets:", len(bdns_assets.items))


=== BDNS Extraction Result ===
BDNS-tagged assets: 6511


In [18]:
# 先頭 N 件の確認出力
N = min(10, len(bdns_assets.items))
for asset in bdns_assets.items[:N]:
    # Contract 仮定: asset.ifc_guid, asset.bdns_tag, asset.name, asset.raw_ifc_class
    print(
        "GUID:", getattr(asset, "ifc_guid", None),
        "| BDNS:", getattr(asset, "bdns_tag", None),
        "| Name:", getattr(asset, "name", None),
        "| Class:", getattr(asset, "raw_ifc_class", None),
    )

GUID: 2HwPth0ED3muP_ET7DKBaz | BDNS: outlet - double switched socket outlet | Name: DSSO-00001 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK99g | BDNS: outlet - double switched socket outlet | Name: DSSO-00002 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9CM | BDNS: outlet - double switched socket outlet | Name: DSSO-00003 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9Ws | BDNS: outlet - double switched socket outlet | Name: DSSO-00004 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9_L | BDNS: outlet - double switched socket outlet | Name: DSSO-00005 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7lq | BDNS: outlet - double switched socket outlet | Name: DSSO-00006 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7qD | BDNS: outlet - double switched socket outlet | Name: DSSO-00007 | Class: IfcFlowTerminal
GUID: 3ID7pTUZ96xeg742Mq2AA7 | BDNS: outlet - double switched socket outlet | Name: DSSO-00008 | Class: IfcFlowTerminal
GUID: 2Rg2Qh64n6AxHfQKUcJx1N | BDNS: out

In [19]:
# ============ 空間要素抽出と JSON 出力 ============
spatial = extract_spatial_elements(bundle)

spatial_json_path = OUT_DIR / "spatial_elements.json"
with open(spatial_json_path, "w", encoding="utf-8") as f:
    json.dump(
        [getattr(e, "__dict__", {}) for e in spatial.items],
        f,
        ensure_ascii=False,
        indent=2,
    )
print("\nspatial_elements.json 出力:", spatial_json_path)


spatial_elements.json 出力: C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\data\output\spatial_elements.json


In [21]:
# ============ Brick Equipment RDF 出力 ============
# Morgate51 のベース名前空間
BASE_NS = "https://id.morgate51.org/project#"

# BrickEquipmentSet → Turtle
equip_ttl_path = OUT_DIR / "equipment.ttl"
write_equipments_turtle(bdns_assets, BASE_NS, equip_ttl_path)
print("equipment.ttl 出力:", equip_ttl_path)

print("\n=== Done ===")


AttributeError: 'BdnsTaggedAsset' object has no attribute 'brick_class'